In [1]:
import math
import pandas as pd

import numpy as np
from scipy.stats import spearmanr
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

In [2]:
df = pd.read_parquet("train_team_track.parquet", engine='fastparquet')

In [ ]:
df.info()

In [ ]:
df.isna().sum()

Нет ни нулевых значений, ни категориальных данных

In [ ]:
df.head()

In [ ]:
variable_features = df.columns[3:]
static_features = df.columns[:2]

In [ ]:
print(f"Количество уникальных route_id: {df['route_id'].nunique()}")
print(f"Количество уникальных office_from_id: {df['office_from_id'].nunique()}")
print(f"Количество уникальных пар (route_id, office_from_id): {df[['route_id', 'office_from_id']].drop_duplicates().shape[0]}")

In [ ]:
routes_with_offices = df.groupby('office_from_id')['route_id'].nunique()
routes_with_offices[routes_with_offices != 1]

Каждому складу принадлежит несколько маршрутов, но каждому маршруту принадлежит один склад

In [ ]:
amount_rows_in_routes_with_office = df.groupby(['office_from_id', 'route_id']).size()
amount_rows_in_routes_with_office[amount_rows_in_routes_with_office != 4342]

Все группы склад-маршрут имеют одинаковый размер

In [ ]:
timestamps = df.groupby(['office_from_id', 'route_id'])['timestamp'].agg(['first', 'last'])
timestamps[(timestamps['first']!=timestamps.iloc[0]['first']) | (timestamps['last']!=timestamps.iloc[0]['last'])]

In [ ]:
print(f"Начальное время отследивания: {timestamps.iloc[0]['first']}")
print(f"Конечное время отследивания: {timestamps.iloc[0]['last']}")

Во всех группах склад-маршрут время началное и конечное время одинаковы

Можно отметить, что даты приходятся на праздники: 8 марта, 1 апреля, 1 мая, 9 мая

In [ ]:
df['day'] = pd.to_datetime(df['timestamp']).dt.dayofweek

In [ ]:
status_cols = ['status_1', 'status_2', 'status_3', 'status_4', 
               'status_5', 'status_6', 'status_7', 'status_8']

fig, axes = plt.subplots(3, 3, figsize=(15, 12))
axes = axes.flatten()

for i, status in enumerate(status_cols):
    sns.boxplot(data=df, x='day', y=status, ax=axes[i])
    axes[i].set_title(status)

plt.tight_layout()
plt.show()

In [ ]:
pd.set_option('display.max_columns', None)
descr = df.groupby('day')[variable_features[:-1]].describe()
descr.index = descr.index.map({0:'Mon', 1:'Tue', 2:'Wed', 3:'Thu', 4:'Fri', 5:'Sat', 6:'Sun'})
print(descr)
pd.set_option('display.max_columns',20)

Сильный разброс, медиана во всех статусах в разы меньше среднего. Но можно отметить, что со вторника по среду наблюдается прирост по всем статусам, а также прирост по таргету. 

In [ ]:
df.iloc[1599299:1599310]

In [ ]:
for i in range(8):
    print(df.iloc[1599299+i:1599299+i+4][variable_features[:-2]].sum().sum())

Видно, что несмотря на сильную убыль количества товара суммарно по всем статусам, значение таргета остаётся неизменным (с 3:00 по 4:00). Либо не все товары, прошедшие конкретные этапы обработки, по итогу оказываются выгруженными. Либо емкость, в которой измеряется target_2h, способна вместить довольно много товаров

In [ ]:
corr = df[['status_1','status_2','status_3','status_4',
           'status_5','status_6','status_7','status_8','target_2h']].corr(method='spearman')
print(corr['target_2h'].sort_values(ascending=False))

Возможно, корреляция не совсем корректна из-за того, что мы учитываем количество товаров, а не разницу между данными, переставшими отслеживаться (из-за того, что "двухчасовое окно", которое отслеживает таргет, сдвинулось), и новыми данными, которые только попали в "двухчасовое окно"

Думаю, стоит посмотреть на общее распределение значений, чтобы увидеть общую картину, но будет отсматривать на десятитысячной выборке, при этом случайно выбирая экземпляры. да, не учитываем время, но для текущей цели посмотреть общее распределение это неважно

In [ ]:
df_sample, _ = train_test_split(
    df, 
    train_size=10000,  
    stratify=df[static_features], 
    random_state=1
)

In [ ]:
def density_hist(feature, df):
    plt.hist(df[feature], bins=50, density=True)
    plt.xlabel(feature)
    plt.ylabel('density')
    plt.show()

In [ ]:
for feature in df_sample.columns[3:-1]:
    density_hist(feature, df_sample)

Никаких артефактов не видно, данные вполне логичны

In [ ]:
corr_df_sample = df_sample[variable_features].corr(method='spearman')

In [ ]:
plt.figure(figsize=(12,10))
ax = sns.heatmap(data=corr_df_sample, annot=True, cmap='Oranges', fmt='.2f')
ax.set_xticklabels(ax.get_xticklabels(), fontsize=10)
ax.set_yticklabels(ax.get_yticklabels(), fontsize=10)
plt.title('truncated dataset')

plt.tight_layout()
plt.show()

In [ ]:
df['hours'] = pd.to_datetime(df['timestamp']).dt.hour + pd.to_datetime(df['timestamp']).dt.minute/60
df['hours']

In [ ]:
def density_dayweek_time():
    df_grouped_dayweek_time = df.groupby(['day', 'hours'])[variable_features[:-1]].mean().reset_index()
    
    _, axes = plt.subplots(9, 1, figsize=(15, 100))
    axes = axes.flatten()

    day_names = {0:'Mon',1:'Tue',2:'Wed',3:'Thu',4:'Fri',5:'Sat',6:'Sun'}

    for i, feature in enumerate(variable_features[:-1]):
        for day in range(7):
            data = df_grouped_dayweek_time[df_grouped_dayweek_time['day'] == day]
            axes[i].plot(data['hours'], data[feature], label=day_names[day])

        axes[i].set_title(feature, fontsize=20)
        axes[i].set_xlabel('Hour', fontsize=16)
        axes[i].set_ylabel('Mean value', fontsize=16)
        axes[i].set_xticks(np.arange(0, 24, 0.5))
        axes[i].set_xticklabels([f'{int(h)}:{int((h%1)*60):02d}' for h in np.arange(0, 24, 0.5)], rotation=45)
        axes[i].legend(fontsize=16)

    plt.tight_layout()
    plt.show()

In [ ]:
density_dayweek_time()

Как видно из графиков, в средних значениях статусов наблюдаются определённые тенденции:
- спад status_1 наблюдается с 0:00 до 5:00, рост с 5:00 до 14:00, потом незначительные колебания
- спад status_2 наблюдается с 0:00 до 4:30, рост с 4:30 до 12:00, потом незначительные колебания
- спад status_3 наблюдается с 0:00 до 7:30, рост с 7:30 до 14:00, потом спад с 14:00 до 0:00 следующего дня
- спад status_4 наблюдается с 0:00 до 8:00, рост с 8:00 до 17:00, потом спад с 17:00 до 21:30, потом рост до 0:00 следующего дня
- спад status_5 наблюдается с 0:00 до 8:00, рост с 8:00 до 16:00, потом спад с 16:00 до 21:00, потом рост до 0:00 следующего дня
- спад status_6 наблюдается с 0:00 до 9:30, рост с 9:40 до 16:30, потом спад с 16:30 до 21:30, потом рост до 0:00 следующего дня
- status_7 и status_8 проявляют себя очень нестабильно, постоянно скачут, но всё равно общий тренд отследить возможно
- у status_7 спад наблюдается с 0:00 до 10:30, причём с 8:30 до 10:30 очень резкий скачок вниз, там значение за 1.5 часа уменьшается на 1000, с 10:30 до 20:30 рост, а потом с 20:30 до 21:30 снова резкий скачок вниз, за 1 час уменьшается на 1000, дальше до 0:00 рост
- у status_8 очень большие колебания, но всё равно можно отследить некий тренд, с 0:00 до 10:30 спад, потом до следующего дня рост
- у target_2h наблюдается спад с 0:00 до 10:30, причём с 8:30 до 10:30 спад более резкий, чем до этого, рост с 10:30 до 18:30, потом снова спад с 18:30 до 23:00, дальше рост до 0:00 следующего дня

### Временные интервалы ключевых изменений
| Статус | Начало спада | Конец спада | Начало роста | Пик активности | Вечерний спад |
|--------|--------------|-------------|--------------|----------------|---------------|
| status_1 | 00:00 | 05:00 | 05:00 | 14:00 | - |
| status_2 | 00:00 | 04:30 | 04:30 | 12:00 | - |
| status_3 | 00:00 | 07:30 | 07:30 | 14:00 | с 14:00 |
| status_4 | 00:00 | 08:00 | 08:00 | 17:00 | 17:00-21:30 |
| status_5 | 00:00 | 08:00 | 08:00 | 16:00 | 16:00-21:00 |
| status_6 | 00:00 | 09:30 | 09:40 | 16:30 | 16:30-21:30 |
| status_7 | 00:00 | 10:30 | 10:30 | 20:30 | 20:30-21:30 |
| status_8 | 00:00 | 10:30 | 10:30 | 00:00 | - |
| target_2h | 00:00 | 10:30 | 10:30 | 18:30 | 18:30-23:00 |

### Сдвиг пиков активности
| Этап | Статусы | Пик активности | Относительно предыдущего |
|------|---------|----------------|--------------------------|
| Поступление | 1-3 | 12:00-14:00 | базовый |
| Обработка | 4-6 | 16:00-17:00 | +2-4 часа |
| Готовность | 7-8 | 20:30 | +4-6 часов |
| Отгрузка | target_2h | 18:30 | +2-4 часа |

In [ ]:
rolling_window_sum_status = pd.DataFrame()
rolling_window_sum_status

In [ ]:
for feature in variable_features[:-1]:
    rolling_window_sum_status[f'status_{feature}_sum'] = df[feature].rolling(window=4).sum()

rolling_window_sum_status['target_2h'] = df['target_2h']
rolling_window_sum_status

In [ ]:
corr = rolling_window_sum_status.iloc[3:].corr(method='spearman')
corr

In [ ]:
plt.figure(figsize=(12,10))
sns.heatmap(corr, cmap='Oranges', annot=True)
plt.show()

In [ ]:
for i in range(1, 8):
        for j in range(i+1, 9):
            lag_corr = df[f'status_{i}'].corr(df[f'status_{j}'].shift(-1))
            print(f"status_{i}(t) → status_{j}(t+1): {lag_corr:.3f}")

# Rough solution without trainable models

In [ ]:
df_temp = df.drop('timestamp', axis=1, inplace=False)

In [ ]:
df_train = pd.DataFrame(columns=df_temp.columns)
df_test = pd.DataFrame(columns=df_temp.columns)

first_test_index = round(4342*0.8)

for group, data in df_temp.groupby(["office_from_id", "route_id"]):
    df_train = pd.concat([df_train, data.reset_index(drop=True).iloc[:first_test_index]], ignore_index=True)
    df_test = pd.concat([df_test, data.reset_index(drop=True).iloc[first_test_index:]], ignore_index=True)
df_train

In [ ]:
df_train['hours'] = pd.to_datetime(df_train['timestamp']).dt.hour + pd.to_datetime(df_train['timestamp']).dt.minute/60
df_test['hours'] = pd.to_datetime(df_test['timestamp']).dt.hour + pd.to_datetime(df_test['timestamp']).dt.minute/60

In [ ]:
df_train_temp = df_train.drop(['office_from_id', 'route_id'], axis=1, inplace=False)
df_train_temp

In [ ]:
df_train_temp.drop('timestamp', axis=1, inplace=True)

In [ ]:
df_mean_value = df_train_temp.groupby('hours').mean()
df_mean_value

In [ ]:
df_test_temp = df_test.drop(['office_from_id', 'route_id'], axis=1, inplace=False)
df_test_temp

In [ ]:
df_temp = df_mean_value.loc[0.0:df_test.iloc[0]['hours']-0.5]
df_mean_value.drop(df_mean_value.loc[0.0:df_test.iloc[0]['hours'] - 0.5].index, inplace=True)
df_mean_value = pd.concat([df_mean_value, df_temp])
df_mean_value

In [ ]:
df_mean_value = df_mean_value.reset_index(drop=True)
n = ((df_test['office_from_id']!=1)|(df_test['route_id']!=33)).idxmax()

df_mean_expanded = df_mean_value.iloc[
    np.tile(np.arange(len(df_mean_value)), n // len(df_mean_value) + 1)
][:n].reset_index(drop=True)

n = df_test_temp.shape[0]

df_mean_expanded = df_mean_expanded.iloc[
    np.tile(np.arange(len(df_mean_expanded)), n // len(df_mean_expanded) + 1)
][:n].reset_index(drop=True)

df_mean_expanded

In [ ]:
rmse = 0;
crit = ['status_1', 'status_2', 'status_3', 'status_4', 'status_5', 'status_6', 'status_7', 'status_8']

for i in range(0, df_test_temp.shape[0]//32, 32):
    rmse += mean_squared_error(df_test_temp[crit].iloc[i:i+32], df_mean_expanded[crit].iloc[i:i+32])
rmse = math.sqrt(rmse/(df_test_temp.shape[0]//32))
rmse

In [ ]:
math.sqrt(mean_squared_error(df_test_temp[crit], df_mean_expanded[crit]))

# Подготовка данных для RNN

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler

In [ ]:
df['hour_sin'] = np.sin(
    2 * np.pi * pd.to_datetime(df['timestamp']).dt.hour + pd.to_datetime(df['timestamp']).dt.minute/60 / 24
)
# df['hour_cos'] = np.cos(
#     2 * np.pi * pd.to_datetime(df['timestamp']).dt.hour + pd.to_datetime(df['timestamp']).dt.minute/60 / 24
# )

df['dow_sin'] = np.sin(2 * np.pi * df['timestamp'].dt.dayofweek / 7)
# df['dow_cos'] = np.cos(2 * np.pi * df['timestamp'].dt.dayofweek / 7)

In [ ]:
df_train = pd.read_csv('df_train.csv')
df_test = pd.read_csv('df_test.csv')

In [ ]:
df_train = pd.DataFrame(columns=df.columns)
df_test = pd.DataFrame(columns=df.columns)

first_test_index = round(4342*0.9)

for group, data in df.groupby(["office_from_id", "route_id"]):
    df_train = pd.concat([df_train, data.reset_index(drop=True).iloc[:first_test_index]], ignore_index=True)
    df_test = pd.concat([df_test, data.reset_index(drop=True).iloc[first_test_index:]], ignore_index=True)
df_train

In [ ]:
scalers = {}

for col in df.columns[3:11]:
    scaler = StandardScaler()
    df_train[col] = scaler.fit_transform(df_train[[col]])
    df_test[col] = scaler.transform(df_test[[col]])
    scalers[col] = scaler

In [ ]:
class WarehouseDataset(Dataset):
    def __init__(self, data, seq_len=4):
        self.seq_len = seq_len
        self.status_cols = [f'status_{i}' for i in range(1, 9)]
        
        self.sequences = []
        self.offices = []
        self.routes = []
        self.hours = []
        self.dow = []
        self.targets = []
        
        for route_id, group in data.groupby('route_id'):
            office_id = group['office_from_id'].iloc[0]
            status_values = group[self.status_cols].values
            hours = group['hour_sin'].values
            dow = group['dow_sin'].values
            
            for i in range(len(status_values) - seq_len):
                self.sequences.append(status_values[i:i+seq_len])
                self.offices.append(office_id)
                self.routes.append(route_id)
                self.dow.append(dow[i:i+seq_len])
                self.hours.append(hours[i:i+seq_len])
                self.targets.append(status_values[i+seq_len])

    def __len__(self):
        return len(self.sequences)
    
    def __getitem__(self, idx):
        seq = torch.FloatTensor(self.sequences[idx])
        office = torch.LongTensor([self.offices[idx] - 1]) 
        route = torch.LongTensor([self.routes[idx]])
        hours = torch.FloatTensor(self.hours[idx])
        dow = torch.FloatTensor(self.dow[idx])
        target = torch.FloatTensor(self.targets[idx])
        
        return seq, office, route, hours, dow, target


In [ ]:
class WarehouseLSTM(nn.Module):
    def __init__(self, input_size=8, hidden_size=64, num_layers=2, office_embedding_dim=8, route_embedding_dim=16):
        super().__init__()
        
        self.office_embedding = nn.Embedding(54, office_embedding_dim)
        self.route_embedding = nn.Embedding(1000, route_embedding_dim)
        
        self.dynamic_context_fc = nn.Linear(2, hidden_size)
        
        static_context_dim = office_embedding_dim + route_embedding_dim
        self.init_hidden_fc = nn.Linear(static_context_dim, hidden_size)
        self.init_cell_fc = nn.Linear(static_context_dim, hidden_size)
        
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, dropout=0.3)
        
        self.fc = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.ReLU(),
#             nn.Dropout(0.2),
            nn.Linear(hidden_size // 2, input_size)
        )
    
    def forward(self, seq, office, route, hours, dow):
        batch_size, seq_len = seq.shape[:2]
        
        office_emb = self.office_embedding(office.squeeze())
        route_emb = self.route_embedding(route.squeeze())
        static_context = torch.cat([office_emb, route_emb], dim=1)
        
        h0 = self.init_hidden_fc(static_context).unsqueeze(0)
        c0 = self.init_cell_fc(static_context).unsqueeze(0)
        
        lstm_out, (h_n, c_n) = self.lstm(seq, (h0, c0))

        dynamic_context = torch.stack([hours, dow], dim=2)
        
        dynamic_effect = self.dynamic_context_fc(dynamic_context)
        
        combined = lstm_out + dynamic_effect
        
        last_lstm = combined[:, -1, :]
        
        output = self.fc(last_lstm)
        
        return output

In [ ]:
def train_model(model, train_loader, val_loader, epochs=30, lr=0.001):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=10, factor=0.5)
    
    for epoch in range(epochs):
        model.train()
        train_loss = 0
        for seq, office, route,  hours, dow, target in train_loader:
            seq = seq.to(device)
            office = office.squeeze().to(device)
            route = route.squeeze().to(device)
            hours = hours.to(device)
            dow = dow.to(device)
            target = target.to(device)
            
            optimizer.zero_grad()
            output = model(seq, office, route, hours, dow)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
        
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for seq, office, route, hours, dow, target in val_loader:
                seq = seq.to(device)
                office = office.squeeze().to(device)
                route = route.squeeze().to(device)
                hours = hours.to(device)
                dow = dow.to(device)
                target = target.to(device)
                
                output = model(seq, office, route, hours, dow)
                val_loss += criterion(output, target).item()
        
        avg_train_loss = train_loss / len(train_loader)
        avg_val_loss = val_loss / len(val_loader)
        
        scheduler.step(avg_val_loss)
        
        print(f'Epoch {epoch}: Train Loss = {math.sqrt(avg_train_loss):.4f}, Val Loss = {math.sqrt(avg_val_loss):.4f}')
    
    return model

In [ ]:
train_dataset = WarehouseDataset(df_train, seq_len=6)
val_dataset = WarehouseDataset(df_test, seq_len=6)

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=False)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

In [ ]:
model = WarehouseLSTM(
        input_size=8,
        hidden_size=64,
        num_layers=1,
        office_embedding_dim=8,
        route_embedding_dim=16
    )

In [ ]:
model = train_model(model, train_loader, val_loader, epochs=30)

In [ ]:
for col in ['route_id', 'office_from_id', 'dow_sin', 'hour_sin']:
    df_train[col] = pd.to_numeric(df_train[col], errors='coerce')
    df_test[col] = pd.to_numeric(df_test[col], errors='coerce')


In [3]:
df_train = pd.DataFrame(columns=df.columns)
df_test = pd.DataFrame(columns=df.columns)

first_test_index = 500
last_test_index = 510

for group, data in df.groupby(["office_from_id", "route_id"]):
    df_train = pd.concat([df_train, data.reset_index(drop=True).iloc[:first_test_index]], ignore_index=True)
    df_test = pd.concat([df_test, data.reset_index(drop=True).iloc[first_test_index:last_test_index]], ignore_index=True)
df_train

,office_from_id,route_id,timestamp,status_1,status_2,status_3,status_4,status_5,status_6,status_7,status_8,target_2h
0,1,33,2025-03-01 00:00:00,9133,646,8341,6584,7707,6458,4793,313,134.0
1,1,33,2025-03-01 00:30:00,8311,584,6745,6200,8038,8562,25547,1012,128.0
2,1,33,2025-03-01 01:00:00,7377,517,7974,5743,7866,9140,980,4167,127.0
3,1,33,2025-03-01 01:30:00,6045,405,7478,5504,8348,7340,0,4162,129.0
4,1,33,2025-03-01 02:00:00,5264,331,7459,5850,8163,9243,20753,406,124.0
...,...,...,...,...,...,...,...,...,...,...,...,...
499995,53,959,2025-03-11 07:30:00,0,34,0,516,1041,1348,0,1538,46.0
499996,53,959,2025-03-11 08:00:00,0,49,0,394,1065,580,17874,5434,52.0
499997,53,959,2025-03-11 08:30:00,0,54,0,396,1531,736,375,1836,35.0
499998,53,959,2025-03-11 09:00:00,0,86,0,359,940,1511,2,3406,24.0


In [1]:
import lightgbm as lgb
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

def prepare_features_for_lgbm(df, seq_len=4):
    features_list = []
    targets_list = []
    
    for route_id, group in df.groupby('route_id'):
        group = group.sort_values('timestamp')
        
        office_id = group['office_from_id'].iloc[0]
        
        status_cols = [f'status_{i}' for i in range(1, 9)]
        status_values = group[status_cols].values
        
        timestamps = pd.to_datetime(group['timestamp'])
        hours = timestamps.dt.hour.values
        minutes = timestamps.dt.minute.values
        day_of_week = timestamps.dt.dayofweek.values
        
        hour_sin = np.sin(2 * np.pi * hours / 24)
        hour_cos = np.cos(2 * np.pi * hours / 24)
        minute_sin = np.sin(2 * np.pi * minutes / 60)
        minute_cos = np.cos(2 * np.pi * minutes / 60)
        dow_sin = np.sin(2 * np.pi * day_of_week / 7)
        dow_cos = np.cos(2 * np.pi * day_of_week / 7)
        
        for i in range(len(status_values) - seq_len):
            features = {}
            
            for t in range(seq_len):
                for s_idx, s_name in enumerate(status_cols):
                    features[f'{s_name}_t{t+1}'] = status_values[i + t, s_idx]
            
            current_hour = hours[i + seq_len]
            current_minute = minutes[i + seq_len]
            current_dow = day_of_week[i + seq_len]
            
            features['hour'] = current_hour
            features['hour_sin'] = np.sin(2 * np.pi * current_hour / 24)
            features['hour_cos'] = np.cos(2 * np.pi * current_hour / 24)
            features['minute'] = current_minute
            features['minute_sin'] = np.sin(2 * np.pi * current_minute / 60)
            features['minute_cos'] = np.cos(2 * np.pi * current_minute / 60)
            features['day_of_week'] = current_dow
            features['dow_sin'] = np.sin(2 * np.pi * current_dow / 7)
            features['dow_cos'] = np.cos(2 * np.pi * current_dow / 7)
            
            features['office_from_id'] = office_id
            features['route_id'] = route_id
            
            for s_idx, s_name in enumerate(status_cols):
                past_values = [status_values[i + t, s_idx] for t in range(seq_len)]
                features[f'{s_name}_mean'] = np.mean(past_values)
                features[f'{s_name}_std'] = np.std(past_values)
                features[f'{s_name}_trend'] = past_values[-1] - past_values[0]
            
            target = status_values[i + seq_len]
            
            features_list.append(features)
            targets_list.append(target)
    
    return pd.DataFrame(features_list), np.array(targets_list)

print("Подготовка признаков для LightGBM...")
X_train, y_train = prepare_features_for_lgbm(df_train, seq_len=4)
X_val, y_val = prepare_features_for_lgbm(df_test, seq_len=4)

print(f"Train shape: {X_train.shape}")
print(f"Val shape: {X_val.shape}")
print(f"Target shape: {y_train.shape}")  # (n_samples, 8)

Подготовка признаков для LightGBM...


NameError: name 'df_train' is not defined

In [7]:
from sklearn.multioutput import MultiOutputRegressor

X_train_flat = X_train
y_train_flat = y_train

base_lgb = lgb.LGBMRegressor(
    objective='regression',
    n_estimators=200,
    learning_rate=0.05,
    num_leaves=31,
    feature_fraction=0.8,
    bagging_fraction=0.8,
    reg_alpha=0.1,
    reg_lambda=0.1,
    random_state=42,
    n_jobs=8
)

multi_model = MultiOutputRegressor(base_lgb, n_jobs=8)

print("Обучение multi-output модели...")
multi_model.fit(X_train, y_train)

predictions_val_multi = multi_model.predict(X_val)

rmse_multi = np.sqrt(mean_squared_error(y_val, predictions_val_multi))
print(f"Multi-output RMSE: {rmse_multi:.4f}")

for i in range(8):
    rmse = np.sqrt(mean_squared_error(y_val[:, i], predictions_val_multi[:, i]))
    print(f"status_{i+1} RMSE: {rmse:.4f}")

Обучение multi-output модели...
Multi-output RMSE: 1646.3152
status_1 RMSE: 214.6383
status_2 RMSE: 34.3557
status_3 RMSE: 195.3862
status_4 RMSE: 264.3216
status_5 RMSE: 341.0105
status_6 RMSE: 617.5707
status_7 RMSE: 4329.6577
status_8 RMSE: 1511.2648


In [8]:
lgb_models = {}
predictions_val = np.zeros((len(X_val), 8))

for i in range(8):
    print(f"\nОбучение модели для status_{i+1}...")
    
    val_data = lgb.Dataset(X_val, label=y_val[:, i], reference=train_data)
    
    params = {
        'objective': 'regression',
        'metric': 'rmse',
        'boosting_type': 'gbdt',
        'num_leaves': 31,
        'learning_rate': 0.05,
        'feature_fraction': 0.8,
        'bagging_fraction': 0.8,
        'bagging_freq': 5,
        'verbose': 0,
        'num_threads': 8,
        'reg_alpha': 0.1,
        'reg_lambda': 0.1,
        'min_child_samples': 20,
        'subsample': 0.8
    }
    
    model = lgb.train(
        params,
        train_data,
        num_boost_round=200,
        valid_sets=[train_data, val_data],
        valid_names=['train', 'val'],
        callbacks=[
            lgb.early_stopping(stopping_rounds=20),
            lgb.log_evaluation(50)
        ]
    )
    
    lgb_models[f'status_{i+1}'] = model
    
    predictions_val[:, i] = model.predict(X_val, num_iteration=model.best_iteration)

rmse_per_status = []
for i in range(8):
    rmse = np.sqrt(mean_squared_error(y_val[:, i], predictions_val[:, i]))
    rmse_per_status.append(rmse)
    print(f"status_{i+1} RMSE: {rmse:.4f}")

total_rmse = np.sqrt(np.mean((y_val - predictions_val) ** 2))
print(f"\nTotal RMSE: {total_rmse:.4f}")


Обучение модели для status_1...
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=0.8 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=0.8 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=0.8 will be ignored. Current value: bagging_fraction=0.8
Training until validation scores don't improve for 20 rounds
[50]	train's rmse: 376.494	val's rmse: 304.53
Early stopping, best iteration is:
[79]	train's rmse: 284.364	val's rmse: 216.336

Обучение модели для status_2...
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=0.8 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=0.8 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=0.8 will be ignored. Current value: bagging_fraction=0.8
Training until validation scores don't impr

In [11]:
y_val

array([[0, 40, 0, ..., 1, 0, 0],
       [0, 45, 0, ..., 6, 0, 0],
       [0, 48, 0, ..., 10, 658, 0],
       ...,
       [1145, 262, 1323, ..., 1399, 0, 0],
       [1150, 285, 1314, ..., 1427, 0, 0],
       [1104, 243, 1415, ..., 866, 2887, 130]], dtype=object)

In [ ]:
predictions_val